# JN6b - Capstone, part 2: join, decompose, score

**Curriculum notebook 6b of 6.** You have your APR (JN6a) and a verified oracle. Now **join** them, **decompose** every unit of difference into a named category, and read the scorecard. The payoff: your own comparison will **rediscover** the limitation you planted back in JN3.

> Clonable + read-only.

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally** (it detects a checkout and skips). On Colab / a bare session it recreates the minimal repo layout under the working directory so the config cell below finds everything unchanged.

In [1]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
_have_repo = (_here/'scripts'/'build_v2').exists() or any((p/'scripts'/'build_v2').exists() for p in _here.parents)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


local repo detected - no fetch needed


## Config

In [2]:
# === CONFIG (clonable) ===
from pathlib import Path
import sys, glob
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PERMIT_GLOB = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
ORACLE_DB   = str(REPO_ROOT / 'databases/hcd_apr_mirror_2026-06-17_fresh.db')
HEADER_ROW  = 7
sys.path.insert(0, str(REPO_ROOT / 'scripts')); sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root:', REPO_ROOT)


repo root: /Users/johngage/berkeley-data


## Rebuild my completions + index the oracle (recap)

In [3]:
import pandas as pd, sqlite3
from collections import defaultdict
from housing_predicates import is_housing, net_units
from s0_keys import normalize_address
from cpra_dedup import extract_master_permit
import housing_rules as hr
def pdate(x):
    d = pd.to_datetime(str(x), errors='coerce'); return d.date() if pd.notna(d) else None
def capn(a):
    try: return hr.to_canonical_apn(a, 'Alameda')
    except: return None
def load(p):
    d = pd.read_excel(p, dtype=str, header=HEADER_ROW); d.columns = [str(c).strip() for c in d.columns]; return d
df = pd.concat([load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)
df = df[df['PermitNumber'].notna()].rename(columns={'Finaled Date':'FinaledDate','Parcel Number':'ParcelNumber'}).copy()
df['isnew'] = df['Work Type'].astype(str).str.strip() == 'New'
df = df[[is_housing(o,u,n,a) for o,u,n,a in zip(df['OccType'],df['UnitsAdded'],df['NumberUnits'],df['ADU'])]]
bld = defaultdict(lambda: {'units': 0.0, 'hasnew': False, 'final': [], 'apns': set()})
for r in df.itertuples(index=False):
    st = r.StreetType; st = '' if (st is None or str(st).strip().lower() == 'nan') else str(st)
    k = normalize_address(f'{r.StreetNumber} {r.StreetName} {st}'.strip())
    if not k.number: continue
    b = bld[(k.number, k.street, k.stype)]
    b['units'] = max(b['units'], net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU))
    if r.isnew: b['hasnew'] = True
    ca = capn(str(r.ParcelNumber));  b['apns'].add(ca) if ca else None
    pn = str(r.PermitNumber)
    if extract_master_permit(pn) == pn and net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU) > 0:
        f = pdate(r.FinaledDate)
        if f: b['final'].append(f)
mine = {k: b for k, b in bld.items() if (b['hasnew'] or b['units'] > 0) and b['final']}   # MY completions

# the oracle's CO rows (fresh, deduped), indexed by address-bucket and by canonical APN
CO = ['CO_ACUTELY_LOW_INCOME_DR','CO_ACUTELY_LOW_INCOME_NDR','CO_EXTREMELY_LOW_INCOME_DR','CO_EXTREMELY_INCOME_NDR',
      'CO_VLOW_INCOME_DR','CO_VLOW_INCOME_NDR','CO_LOW_INCOME_DR','CO_LOW_INCOME_NDR','CO_MOD_INCOME_DR','CO_MOD_INCOME_NDR','CO_ABOVE_MOD_INCOME']
S = '+'.join(f"CAST(NULLIF({c},'') AS INT)" for c in CO)
con = sqlite3.connect(f'file:{ORACLE_DB}?mode=ro', uri=True)
obucket, oapn, ofull, seen = defaultdict(list), defaultdict(list), set(), set()
for apn, addr, co, u in con.execute(f"SELECT APN,STREET_ADDRESS,CO_ISSUE_DT1,({S}) u FROM table_a2 WHERE CO_ISSUE_DT1<>''"):
    if (apn, addr, co, u) in seen: continue
    seen.add((apn, addr, co, u)); y = int(co[:4]) if co[:4].isdigit() else None
    if not y: continue
    ak = normalize_address(addr)
    obucket[ak.bucket].append((ak, y, u or 0)); ofull.add((ak.number, ak.street, ak.stype))
    ca = capn(apn)
    if ca: oapn[ca].append((y, u or 0))
print(f'{len(mine)} of my completions vs the oracle CO rows')


951 of my completions vs the oracle CO rows


# PHASE 6c - Join & compare

To compare, each of *my* completed buildings must find its *oracle* row. The trap (JN2): a naive string-`==` join fails on the suffix mismatch (`<2650 TELEGRAPH>` vs `<...AVE>`) and **under-matches** - making real matches look like gaps. We use the **`matches()` relation** (suffix-tolerant) and then **augment with the canonical APN** for the few the address still misses. Watch how many each layer recovers over the naive join.

In [4]:
string_only = matches_recovered = apn_recovered = unmatched = 0
for k, b in mine.items():
    vk = normalize_address(f'{k[0]} {k[1]} {k[2] or ""}'.strip())
    bucket_cands = [o for o in obucket.get(vk.bucket, []) if vk.matches(o[0])]
    string_hit = (vk.number, vk.street, vk.stype) in ofull         # would a naive == have matched?
    apn_hit = any(a in oapn for a in b['apns'])
    if bucket_cands and string_hit: string_only += 1
    elif bucket_cands:              matches_recovered += 1          # matches() caught a suffix mismatch
    elif apn_hit:                   apn_recovered += 1              # APN caught what the address missed
    else:                           unmatched += 1
matched_core = string_only + matches_recovered + apn_recovered
print(f'naive string== would match : {string_only}')
print(f'  + matches() recovered     : {matches_recovered}   (suffix mismatches a == join drops)')
print(f'  + APN-augment recovered   : {apn_recovered}')
print(f'  = matched core            : {matched_core}    (unmatched: {unmatched})')

naive string== would match : 370
  + matches() recovered     : 320   (suffix mismatches a == join drops)
  + APN-augment recovered   : 11
  = matched core            : 701    (unmatched: 250)


### Checkpoint 6c

In [5]:
assert string_only == 370 and matches_recovered == 320 and apn_recovered == 11
assert matched_core == 701
print('CHECKPOINT 6c PASS')
print(f'  matched core {matched_core}; the relation+APN recovered {matches_recovered + apn_recovered} matches a naive == would have called gaps')

CHECKPOINT 6c PASS
  matched core 701; the relation+APN recovered 331 matches a naive == would have called gaps


# PHASE 6d - Decompose & score

Now account for **every unit** of the my-vs-city difference, by year, into named buckets:

- **v3-found** - units I have that the city's A2 lacks at that address (real found completions);
- **city-coverage** - units the city has that I lack (a genuine coverage gap);
- **matched-diff** - on shared addresses, the net unit difference.

The net must reconcile to (my total - city deduped total).

In [6]:
mine_by = defaultdict(int); city_by = defaultdict(int)
for k, b in mine.items(): mine_by[(k[:2], max(b['final']).year)] += int(b['units'])
for bk, rows in obucket.items():
    for ak, y, u in rows: city_by[(bk, y)] += u
found = coverage = matched_diff = 0
for key in set(mine_by) | set(city_by):
    v, c = mine_by.get(key, 0), city_by.get(key, 0)
    if v and c: matched_diff += v - c
    elif v:     found += v
    elif c:     coverage += c
net = found - coverage + matched_diff
print(f'  v3-found (city lacks)   : +{found}')
print(f'  city-coverage (I lack)  : -{coverage}')
print(f'  matched-diff            : {matched_diff:+}')
print(f'  ----------------------------------')
print(f'  NET (my total - city)   : {net:+}')

  v3-found (city lacks)   : +573
  city-coverage (I lack)  : -350
  matched-diff            : +78
  ----------------------------------
  NET (my total - city)   : +301


## The capstone rediscovery: the residual surfaces 2352 Shattuck

Drill into the difference and one address jumps out - **2352 Shattuck**. Look at how the **city** records it versus how **my APR** does:

In [7]:
k = normalize_address('2352 Shattuck Ave')
mine_row = mine[(k.number, k.street, k.stype)]
city_rows = [(ak.raw, y, u) for ak, y, u in obucket[k.bucket]]
print('city records 2352 Shattuck as:')
for raw, y, u in sorted(city_rows, key=lambda r: r[1]):
    print(f'    {u:>4} units  @ {y}   ({raw})')
print(f'my APR records it as:')
print(f'    {int(mine_row["units"]):>4} units  @ {max(mine_row["final"]).year}   (ONE building)')

city records 2352 Shattuck as:
     135 units  @ 2022   (2352 SHATTUCK)
      69 units  @ 2023   (2352 Shattuck Ave)
my APR records it as:
     135 units  @ 2023   (ONE building)


### Name it - your own scorecard found the limitation you planted

The city has **two** buildings: a **135-unit North @ 2022** and a **69-unit South @ 2023**. My APR has **one** 135-unit building @ 2023. This is exactly the limitation flagged in **JN3** (address-grouping collapses two-buildings-at-one-address) and **JN4** (so the collapsed building took the South's 2023 date) - and now your **own scorecard** surfaces it as a real discrepancy: a **+69-unit** miss (the hidden South) and a **year-misattribution** (the North's 135 units belong in 2022, not 2023).

**Fix vs flag vs defer - the informed call.** The fix is known: split by APN in the spine (JN3). But the split does not *propagate* - every downstream stage keys buildings by **address**, so they would re-merge the two unless the whole pipeline is re-keyed to a per-building identity. That re-architecture is real work for **one** building (the only such case in the data). So the honest engineering call is an **informed flag**: document it precisely, quantify it (+69u, one year shift), and defer the re-key - *sometimes flag-and-quantify is the right answer, not fix-at-all-costs.* The lesson is that you can only make that call because you **found and named** it.

### Checkpoint 6d

In [8]:
assert net == found - coverage + matched_diff          # the decomposition reconciles
assert sum(mine_by.values()) == 4310                   # my total
# 2352 Shattuck rediscovered: city splits it, I collapsed it
assert len(city_rows) == 2 and {u for _, _, u in city_rows} == {135, 69}
assert int(mine_row['units']) == 135 and max(mine_row['final']).year == 2023
print('CHECKPOINT 6d PASS')
print(f'  scorecard nets out: found +{found} / coverage -{coverage} / matched {matched_diff:+} = {net:+}')
print('  2352 Shattuck rediscovered + named: city 135@2022 + 69@2023  vs  my collapsed 135@2023')
print('  the JN3 grouping limitation, found by the student\'s own comparison -> informed flag')

CHECKPOINT 6d PASS
  scorecard nets out: found +573 / coverage -350 / matched +78 = +301
  2352 Shattuck rediscovered + named: city 135@2022 + 69@2023  vs  my collapsed 135@2023
  the JN3 grouping limitation, found by the student's own comparison -> informed flag


# Capstone complete

You walked from a raw, human-formatted permit export to a **scored, honest APR**: ingested and verified (JN1), keyed places by a *relation* (JN2), counted units with the *corrected* signal (JN3), derived completion *stages* from dated events (JN4), tagged the *three* date-concepts without conflating them (JN5), and **scored your APR against the city's own report - rediscovering, by your own decomposition, the one limitation you planted along the way** (JN6). Every number traceable; every blank honest; every difference named. That is the whole method.